# RAG Pipeline - Exp1

* The goal is to finish multiple RAG pipelines in fastest way and accurately 🎯
* Compare LlamaIndex RAG performance in different settings
* Save the configs file for each pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
import pandas as pd
import json

from utils import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows


### Load Testing Dataset

In [2]:
# Load raw data with smalll selected testset
dataset = load_dataset("PatronusAI/financebench")
selected_items = dataset['train'].select(range(5))

selected_docs =set()
for dct in selected_items:
    selected_docs.add(f'pdfs/{dct["doc_name"]}.pdf')
print(len(selected_docs))

# Load selected pdfs
loaded_pdf = {}
with ThreadPoolExecutor(max_workers=5) as executor:
    loaded_pdf = dict(executor.map(load_pdf_content, list(selected_docs)))
print(f'{len(loaded_pdf)} pdfs loaded')


# Genrate LlamaIndex Documents from each selected item
documents = []
selected_metadata_cols = ['company', 'doc_name', 'doc_period', 'doc_link']

documents = process_items_parallel(
    selected_items,
    selected_doc_names=set([selected_doc.replace('pdfs/', '').replace('.pdf', '')
                             for selected_doc in selected_docs]),
    loaded_pdf=loaded_pdf,
    selected_metadata_cols=selected_metadata_cols,
    max_workers=10
)
print(f"Generated {len(documents)} documents")

2
2 pdfs loaded
Generated 5 documents


### Run Pipeline with Different Settings

In [3]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

#### Sequentially Run Embedding & Saving Indexing

* Only need to run once for first time generation

In [ ]:
for i in range(len(all_config['embedding_models'])):
    embed_model_str = all_config['embedding_models'][i]
    indexing_storage_dir = all_config['indexing_storage_dirs'][i]
    print(f"Embedding model: {embed_model_str}")
    create_vector_index(documents, embed_model_str, indexing_storage_dir)
    

Embedding model: BAAI/bge-small-en-v1.5
./storage_m1
Embedding model: BAAI/bge-large-en-v1.5
./storage_m2
Embedding model: thenlper/gte-large
./storage_m3


#### Split Config File --> Per Model Per Config

In [5]:
retriever_params = all_config['retriever_params']
cfgs = []
for i in range(len(all_config['yaml_config_files'])):
    yaml_config_file = 'output/saved_configs/' + all_config['yaml_config_files'][i]
    cfgs.append(yaml_config_file)
    config = {
        'llm_model': all_config['llm_models'][i],
        'embedding_model': all_config['embedding_models'][i],
        'indexing_storage_dir': all_config['indexing_storage_dirs'][i],
        'output_file': all_config['output_files'][i],
        'retriever_params': retriever_params
    }

    with open(yaml_config_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

#### Run Rest of Rag Pipelines

In [6]:
max_worker = os.cpu_count() - 1

await run_all_in_processes(cfgs, selected_items, documents, max_workers=max_worker)

### Evaluation

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
import nest_asyncio
nest_asyncio.apply()

eval_llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key= os.getenv("GEMINI_API_KEY")
)

with open('eval_prompts.yaml', 'r') as file:
    prompt_versions = yaml.safe_load(file)

In [3]:
with open("output/m1.json", "r", encoding="utf-8") as f:
    m1_data = json.load(f)
with open("output/m2.json", "r", encoding="utf-8") as f:
    m2_data = json.load(f)
with open("output/m3.json", "r", encoding="utf-8") as f:
    m3_data = json.load(f)

m1_example = m1_data[0]
print(json.dumps(m1_example, indent=2, ensure_ascii=True))

{
  "question": "What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.",
  "expected_answer": "$1577.00",
  "ai_answer": "Unfortunately, the provided context information does not include the FY2018 capital expenditure amount for 3M. The information only includes data for the years 2022, 2021, and 2020.",
  "retrieved_lst": [
    {
      "metadata": "3M_2022_10K",
      "content": "Supplemental Cash Flow Information\n(Millions)\n2022\n2021\n2020\nCash income tax payments, net of refunds\n$\n1,320\n \n$\n1,695\n \n$\n1,351\n \nCash interest payments\n440\n \n472\n \n524\n \nCash interest payments include interest paid on debt and finance lease balances. Cash interest payments exclude the cash paid for early debt extinguishment costs. Additional\ndetails are described in Note 12.\nIndividual amounts in the Consolidated Statement of Cash Flows exclude the impacts of acquisitions, d

In [4]:
m1_eval_input = get_eval_input(m1_data)
print(m1_eval_input.shape)
display(m1_eval_input.head())

m2_eval_input = get_eval_input(m2_data)
print(m2_eval_input.shape)

m3_eval_input = get_eval_input(m3_data)
print(m3_eval_input.shape)

(5, 4)


,query,ai_answer,referenced_answer,retrieved_content
0,What is the FY2018 capital expenditure amount ...,"Unfortunately, the provided context informatio...",$1577.00,Supplemental Cash Flow Information\n(Millions)...
1,Assume that you are a public equities analyst....,To calculate the net PPNE (Present Value of No...,$8.70,"31,\n \nAsset Class\n \n2018\n \n2017\n ..."
2,Is 3M a capital-intensive business based on FY...,"Based on the provided information, it's diffic...","No, the company is managing its CAPEX and Fixe...",The Company expects 2019 capital spending to b...
3,What drove operating margin change as of FY202...,"Based on the provided information, it's challe...",Operating Margin for 3M in FY2022 has decrease...,Additional discussion related to the component...
4,"If we exclude the impact of M&A, which segment...","Based on the information provided, it appears ...",The consumer segment shrunk by 0.9% organically.,Asia Pacific included China/Hong\nKong net sal...


(5, 4)
(5, 4)


In [ ]:
m1_answer_accuracy = asyncio.run(get_answer_accuracy_output_async(m1_eval_input, eval_llm,
                                                        prompt_versions['ac_prompt_template']))
m2_answer_accuracy = asyncio.run(get_answer_accuracy_output_async(m2_eval_input, eval_llm,
                                                        prompt_versions['ac_prompt_template']))
m3_answer_accuracy = asyncio.run(get_answer_accuracy_output_async(m3_eval_input, eval_llm,
                                                        prompt_versions['ac_prompt_template']))

m1_answer_accuracy.to_csv("output/m1_answer_accuracy.csv", index=False)
m2_answer_accuracy.to_csv("output/m2_answer_accuracy.csv", index=False)
m3_answer_accuracy.to_csv("output/m3_answer_accuracy.csv", index=False)
print(m1_answer_accuracy.shape, m2_answer_accuracy.shape, m3_answer_accuracy.shape)
display(m1_answer_accuracy)

(5, 6) (5, 6) (5, 6)


,query,ai_answer,referenced_answer,retrieved_content,answer_accuracy_score,ac_reasoning
0,What is the FY2018 capital expenditure amount ...,"Unfortunately, the provided context informatio...",$1577.00,Supplemental Cash Flow Information\n(Millions)...,1,"AI's ANSWER claims the data is unavailable, wh..."
1,Assume that you are a public equities analyst....,To calculate the net PPNE (Present Value of No...,$8.70,"31,\n \nAsset Class\n \n2018\n \n2017\n ...",1,AI's ANSWER is relevant to the USER QUERY but ...
2,Is 3M a capital-intensive business based on FY...,"Based on the provided information, it's diffic...","No, the company is managing its CAPEX and Fixe...",The Company expects 2019 capital spending to b...,1,AI's ANSWER is relevant but inaccurate. It fai...
3,What drove operating margin change as of FY202...,"Based on the provided information, it's challe...",Operating Margin for 3M in FY2022 has decrease...,Additional discussion related to the component...,1,AI's ANSWER fails to provide the specific driv...
4,"If we exclude the impact of M&A, which segment...","Based on the information provided, it appears ...",The consumer segment shrunk by 0.9% organically.,Asia Pacific included China/Hong\nKong net sal...,1,AI's ANSWER identifies the Health Care segment...


#### Eval Comparison

In [7]:
m1_answer_accuracy = pd.read_csv("output/m1_answer_accuracy.csv")
m2_answer_accuracy = pd.read_csv("output/m2_answer_accuracy.csv")
m3_answer_accuracy = pd.read_csv("output/m3_answer_accuracy.csv")
print(m1_answer_accuracy.shape, m2_answer_accuracy.shape, m3_answer_accuracy.shape)
display(m1_answer_accuracy)

(5, 6) (5, 6) (5, 6)


,query,ai_answer,referenced_answer,retrieved_content,answer_accuracy_score,ac_reasoning
0,What is the FY2018 capital expenditure amount ...,"Unfortunately, the provided context informatio...",$1577.00,Supplemental Cash Flow Information\n(Millions)...,1,"AI's ANSWER claims the data is unavailable, wh..."
1,Assume that you are a public equities analyst....,To calculate the net PPNE (Present Value of No...,$8.70,"31,\n \nAsset Class\n \n2018\n \n2017\n ...",1,AI's ANSWER is relevant to the USER QUERY but ...
2,Is 3M a capital-intensive business based on FY...,"Based on the provided information, it's diffic...","No, the company is managing its CAPEX and Fixe...",The Company expects 2019 capital spending to b...,1,AI's ANSWER is relevant but inaccurate. It fai...
3,What drove operating margin change as of FY202...,"Based on the provided information, it's challe...",Operating Margin for 3M in FY2022 has decrease...,Additional discussion related to the component...,1,AI's ANSWER fails to provide the specific driv...
4,"If we exclude the impact of M&A, which segment...","Based on the information provided, it appears ...",The consumer segment shrunk by 0.9% organically.,Asia Pacific included China/Hong\nKong net sal...,1,AI's ANSWER identifies the Health Care segment...


In [8]:
def add_suffix_except(df, suffix, exclude):
    return df.rename(columns={col: col + suffix for col in df.columns if col not in exclude})
exclude_cols = ['query', 'referenced_answer']
m1 = add_suffix_except(m1_answer_accuracy, '_m1', exclude_cols)
m2 = add_suffix_except(m2_answer_accuracy, '_m2', exclude_cols)
m3 = add_suffix_except(m3_answer_accuracy, '_m3', exclude_cols)

compare_df = m1.merge(m2, on=exclude_cols).merge(m3, on=exclude_cols)
compare_df = compare_df[['query', 'referenced_answer', 'ai_answer_m1', 'ai_answer_m2', 'ai_answer_m3',
                         'answer_accuracy_score_m1', 'answer_accuracy_score_m2', 'answer_accuracy_score_m3',
                         'ac_reasoning_m1', 'ac_reasoning_m2', 'ac_reasoning_m3']]
print(compare_df.shape)
compare_df.head()

(5, 11)


,query,referenced_answer,ai_answer_m1,ai_answer_m2,ai_answer_m3,answer_accuracy_score_m1,answer_accuracy_score_m2,answer_accuracy_score_m3,ac_reasoning_m1,ac_reasoning_m2,ac_reasoning_m3
0,What is the FY2018 capital expenditure amount ...,$1577.00,"Unfortunately, the provided context informatio...",The cash‑flow excerpt provided lists financing...,The 2018 cash‑flow details provided do not inc...,1,1,1,"AI's ANSWER claims the data is unavailable, wh...",The AI's answer is relevant but inaccurate. It...,AI's ANSWER claims the information is not avai...
1,Assume that you are a public equities analyst....,$8.70,To calculate the net PPNE (Present Value of No...,The year‑end FY 2018 net PPNE for 3M is about ...,Year‑end FY2018 net PPNE for 3M: **$6.17 billi...,1,1,1,AI's ANSWER is relevant to the USER QUERY but ...,"AI's ANSWER is relevant to the USER QUERY, but...","AI's ANSWER is relevant to the USER QUERY, but..."
2,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...","Based on the provided information, it's diffic...",I don’t have the FY 2022 figures needed to eva...,The information supplied covers capital‑spendi...,1,1,1,AI's ANSWER is relevant but inaccurate. It fai...,The AI's answer is relevant to the query but f...,"The AI's answer claims a lack of 2022 data, wh..."
3,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,"Based on the provided information, it's challe...",I don’t have information on the specific facto...,The information supplied focuses on 2017 and 2...,1,1,1,AI's ANSWER fails to provide the specific driv...,AI's ANSWER fails to provide the specific driv...,AI's ANSWER claims it lacks information for 20...
4,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,"Based on the information provided, it appears ...",The available information does not include det...,"I’m sorry, but the information you’ve provided...",1,1,1,AI's ANSWER identifies the Health Care segment...,AI's ANSWER claims the information is unavaila...,AI's ANSWER claims it lacks information about ...
